In [1]:
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import KMeans
from pathlib import Path
from IPython.display import display
from tqdm import tqdm

# Install sentence-transformers in the notebook environment if it's missing.
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

nltk.download("all")

Note: you may need to restart the kernel to use updated packages.


/home/h00pyfr00d/Documents/AML_Spotify/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    | 

True

In [2]:
data = pd.read_csv("data/additional_features.csv")
# remove songs with missing lyrics and reset the index
data = data.dropna(subset=['lyrics']).reset_index(drop=True)
data

,id,Rank,Title,Primary Artists,Featuring Artists,Date,Danceability,Energy,Loudness,Speechiness,...,lyrics,detected_languages,num_languages,language_confidences,artist_id,explicit,track_popularity,artist_followers,artist_genres,artist_popularity
0,47EiUVwUp4C9fGccaPuUCS,1,dákiti,"bad bunny, jhayco",NaN,26/11/2020,0.731,0.573,-10059.00,0.054,...,"Baby, ya yo me enteré, se nota cuando me ve'\n...",SPANISH,1,SPANISH: 1.0,4q3ewBCX7sLwd24euuV69X,True,78,102185553.0,"reggaeton, trap latino, urbano latino, latin",95.0
1,7igeByaBM0MgGsgXtNxDJ7,1,positions,ariana grande,NaN,29/10/2020,0.736,0.802,-4759.00,0.086,...,Heaven sent you to me\nI'm just hopin’ I don't...,ENGLISH,1,ENGLISH: 1.0,66CXWjxzNUsdJxJ2JdwvnR,True,0,107261237.0,pop,90.0
2,3qQbCzHBycnDpGskqOWY0E,1,ella baila sola,"eslabon armado, peso pluma",NaN,29/05/2023,0.668,0.758,-5176.00,0.033,...,"Compa, ¿qué le parece esa morra?\nLa que anda ...",SPANISH,1,SPANISH: 1.0,0XeEobZplHxzM9QzFQWLiR,False,73,6434994.0,"corridos tumbados, sad sierreño, corrido, sier...",75.0
3,0Yde1VrWbGF2Tus2AQhMuT,1,circles,post malone,NaN,06/09/2019,0.703,0.756,-3407.00,0.038,...,"Oh, oh, oh\nOh, oh, oh\nOh, oh, oh, oh, oh\n...",ENGLISH,1,ENGLISH: 1.0,246dkjvS1zLTtiykXe5h60,False,0,47686394.0,NaN,85.0
4,0TK2YIli7K1leLovkQiNik,1,señorita,"camila cabello, shawn mendes",NaN,25/07/2019,0.759,0.540,-6039.00,0.029,...,I love it when you call me señorita\nI wish I ...,ENGLISH,1,ENGLISH: 1.0,7n2wHs1TKAczGzO7Dd2rGr,False,79,45665812.0,NaN,81.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8029,7wBJfHzpfI3032CSD7CE2m,200,stargazing,travis scott,NaN,14/05/2020,0.487,0.789,-3.69,0.044,...,"Ah\n\nRollin', rollin', rollin', got me starga...",ENGLISH,1,ENGLISH: 1.0,0Y5tJX1MQlPlqiwlOH1tJY,True,73,41006554.0,rap,89.0
8030,7yotKA30dwTKNEGomV9ZsI,200,kod,j. cole,NaN,28/06/2018,0.823,0.783,-6274.00,0.154,...,"Go, go, go\nGo, go, go\nGo, go, go\nGo, go, go...",ENGLISH,1,ENGLISH: 1.0,6l3HvQ5sa6mXTsMTB19rO5,True,60,27037715.0,rap,82.0
8031,5FOCkGt7bp22nBVfQ4WSIT,200,indecente,anitta,NaN,19/04/2018,0.874,0.602,-5118.00,0.127,...,"Una mirada te delata, un perfume que te atrapa...",SPANISH,1,SPANISH: 1.0,7FNnA9vBm6EKceENgCGRMb,False,48,13245754.0,"brazilian pop, funk pop, brazilian funk, funk ...",78.0
8032,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,27/09/2021,0.923,0.533,-6016.00,0.262,...,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0,7jVv8c5Fj3E9VhNjxT4snq,True,50,12311640.0,NaN,71.0


In [3]:
# Build semantic embeddings for lyrics, find similar songs, and train a simple prediction model.
# Uses sentence-transformers if available, otherwise falls back to TF-IDF.
lyrics = data['lyrics'].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
embeddings_sparse = vectorizer.fit_transform(lyrics)
embeddings = embeddings_sparse  # keep sparse matrix
embeddings_is_sparse = True

nn = NearestNeighbors(n_neighbors=6, metric='cosine', n_jobs=-1)
nn.fit(embeddings)

def get_similar(song_idx: int, k: int = 5):
    """
    Return k most similar songs (index, distance, title, primary artists).
    Excludes the query itself from the results.
    """
    query = embeddings[song_idx] if embeddings_is_sparse else embeddings[song_idx]
    distances, indices = nn.kneighbors([query] if not embeddings_is_sparse else query, n_neighbors=k+1)
    distances = distances[0]
    indices = indices[0]
    results = []
    for dist, idx in zip(distances, indices):
        if idx == song_idx:
            continue
        row = data.iloc[idx]
        results.append({'idx': int(idx), 'distance': float(dist), 'title': row['Title'], 'artists': row['Primary Artists']})
        if len(results) >= k:
            break 
    return results

In [4]:
def get_songs_by_artist(artist: str) -> pd.DataFrame:
    a = artist.strip().lower()
    prim = data['Primary Artists'].fillna('').astype(str)
    feat = data['Featuring Artists'].fillna('').astype(str)

    pattern = re.escape(a)
    mask_prim = prim.str.lower().str.contains(pattern, na=False)
    mask_feat = feat.str.lower().str.contains(pattern, na=False)

    return data[mask_prim | mask_feat].copy()

def get_lyrics_by_title(title: str, exact: bool = True, first_only: bool = True):
    t = title.strip().lower()
    titles = data['Title'].fillna('').astype(str).str.lower()
    if exact:
        mask = titles == t
    else:
        mask = titles.str.contains(re.escape(t), na=False)
    matches = data[mask]

    if matches.empty:
        return None if first_only else matches
    if first_only:
        row = matches.iloc[0]
        return row['lyrics']
    
    return matches[['lyrics']].copy()

features = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

In [ ]:
# Compute/Load semantic embeddings (sentence-transformers) and rank by semantic + feature similarity.
SEM_EMB_PATH = Path("data/semantic_embeddings.npy")
lyrics_list = data['lyrics'].fillna("").astype(str).tolist()

def ensure_semantic_embeddings(model_name: str = "all-MiniLM-L6-v2", force_recompute: bool = False) -> np.ndarray:
    global embeddings_dense, embeddings_is_sparse

    # If already in memory and not forced, return it
    if 'embeddings_dense' in globals() and not force_recompute:
        return embeddings_dense

    # Load from disk if available
    if SEM_EMB_PATH.exists() and not force_recompute:
        embeddings_dense = np.load(SEM_EMB_PATH)
        embeddings_is_sparse = False
        return embeddings_dense

    model = SentenceTransformer(model_name)
    # encode with batched inference; returns numpy array
    embeddings_dense = model.encode(lyrics_list, show_progress_bar=True, convert_to_numpy=True)
    np.save(SEM_EMB_PATH, embeddings_dense)
    embeddings_is_sparse = False
    return embeddings_dense

def get_hybrid_similarity_chunked(liked_idx, tfidf_weight=0.5, semantic_weight=0.5, chunk_size=500):
    n_songs = len(data)
    similarities = np.zeros(n_songs)
    
    # Get liked song TF-IDF embeddings (keep sparse for memory efficiency)
    liked_tfidf = embeddings[liked_idx]
    
    # Get liked song semantic embeddings
    semantic_emb = ensure_semantic_embeddings()
    liked_semantic = semantic_emb[liked_idx]
    
    # Process in chunks to avoid memory issues
    for start_idx in range(0, n_songs, chunk_size):
        end_idx = min(start_idx + chunk_size, n_songs)
        
        # Get TF-IDF similarities for this chunk
        chunk_tfidf = embeddings[start_idx:end_idx]
        tfidf_sims = cosine_similarity(liked_tfidf, chunk_tfidf)
        tfidf_sim_agg = np.median(tfidf_sims, axis=0)
        
        # Get semantic similarities for this chunk  
        chunk_semantic = semantic_emb[start_idx:end_idx]
        semantic_sims = cosine_similarity(liked_semantic, chunk_semantic)
        semantic_sim_agg = np.median(semantic_sims, axis=0)
        
        # Combine similarities with weights
        chunk_combined = tfidf_weight * tfidf_sim_agg + semantic_weight * semantic_sim_agg
        similarities[start_idx:end_idx] = chunk_combined
    
    return similarities

def get_hybrid_embeddings_sample(sample_indices, tfidf_weight=0.5, semantic_weight=0.5):
    # Get TF-IDF embeddings for sample (convert to dense only for sample)
    tfidf_sample = embeddings[sample_indices]
    if hasattr(tfidf_sample, 'toarray'):
        tfidf_dense = tfidf_sample.toarray()
    else:
        tfidf_dense = tfidf_sample
    
    # Get semantic embeddings for sample
    semantic_emb = ensure_semantic_embeddings()
    semantic_sample = semantic_emb[sample_indices]
    
    # Normalize both to unit length
    tfidf_normalized = normalize(tfidf_dense, norm='l2')
    semantic_normalized = normalize(semantic_sample, norm='l2')
    
    # Use smaller dimension to save memory (truncate larger one)
    min_dim = min(tfidf_normalized.shape[1], semantic_normalized.shape[1], 1000)
    tfidf_truncated = tfidf_normalized[:, :min_dim]
    semantic_truncated = semantic_normalized[:, :min_dim]
    
    # Combine with weights and renormalize
    hybrid_embeddings = (tfidf_weight * tfidf_truncated + semantic_weight * semantic_truncated)
    hybrid_embeddings = normalize(hybrid_embeddings, norm='l2')
    
    return hybrid_embeddings

def get_ranked_recommendations(song_list: pd.DataFrame, n_recs: int = 10, model_type="semantic", 
                             semantic_weight: float = 0.7, hybrid_tfidf_weight: float = 0.5) -> pd.DataFrame:
    if song_list.empty:
        return pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists', 'Feature_Similarity', 'Semantic_Similarity', 'Combined_Score'])

    # Impute & scale audio features on full dataset
    impute_vals = data[features].median()
    X_full_imputed = data[features].fillna(impute_vals)
    scaler = StandardScaler().fit(X_full_imputed)
    X_scaled = pd.DataFrame(scaler.transform(X_full_imputed), index=X_full_imputed.index, columns=features)

    liked_idx = song_list.index.to_numpy()
    liked_feat = X_scaled.loc[liked_idx].values
    feat_dists = cosine_distances(liked_feat, X_scaled.values)
    feat_sims_matrix = 1 - feat_dists
    feat_sim_agg = np.median(feat_sims_matrix, axis=0)

    # Word similarities - initialize with zeros as default
    semantic_sim_agg = np.zeros(len(data))
    
    # Hybrid: Use combination of TF-IDF and Semantic embeddings (memory-efficient)
    if model_type == 'hybrid':
        semantic_sim_agg = get_hybrid_similarity_chunked(
            liked_idx, 
            tfidf_weight=hybrid_tfidf_weight, 
            semantic_weight=1.0 - hybrid_tfidf_weight
        )
    
    # Semantic: Use semantic word-embeddings for lyrical similarity
    elif model_type == 'semantic':
        try:
            emb_dense = ensure_semantic_embeddings()
            liked_emb = emb_dense[liked_idx]
            sem_dists = cosine_distances(liked_emb, emb_dense)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        except Exception as e:
            print(f"Semantic embeddings unavailable; using zeros. Error: {str(e)}")
            semantic_sim_agg = np.zeros(len(data))

    # TF-IDF: Use word frequencies for lyrical similarity
    elif model_type == "tf-idf":
        if 'embeddings' in globals() and embeddings is not None:
            liked_emb = embeddings[liked_idx]
            sem_dists = cosine_distances(liked_emb, embeddings)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        else:
            semantic_sim_agg = np.zeros(len(data))

    # Features: Use only the audio features, no lyrical similarity
    elif model_type == 'features':
        semantic_sim_agg = np.zeros(len(data))

    # Random: Use random lyrical similarity
    elif model_type == 'random':
        np.random.seed(42)
        semantic_sim_agg = np.random.rand(len(data))

    # For hybrid model, semantic_weight is ignored (combination already done in embeddings)
    if model_type == 'hybrid':
        combined = 0.7 * semantic_sim_agg + 0.3 * feat_sim_agg  # Default weighting
    else:
        combined = semantic_weight * semantic_sim_agg + (1.0 - semantic_weight) * feat_sim_agg

    scores = pd.DataFrame({
        'Feature Similarity': feat_sim_agg,
        'Lyric Similarity': semantic_sim_agg,
        'Combined Score': combined
    }, index=X_scaled.index)

    # Exclude liked songs robustly (works for any index type) and pick top n_recs
    mask = ~scores.index.isin(liked_idx)
    top = scores.loc[mask].sort_values('Combined Score', ascending=False).head(n_recs)

    recs = data.loc[top.index, ['Rank', 'Title', 'Primary Artists', 'Featuring Artists'] + features].copy()
    recs['Feature Similarity'] = top['Feature Similarity'].values
    recs['Lyric Similarity'] = top['Lyric Similarity'].values
    recs['Combined Score'] = top['Combined Score'].values

    recs = recs.reset_index().drop(columns=['index'])
    return recs

In [6]:
artist = "drake"
artist_songs = get_songs_by_artist(artist)
liked_songs = artist_songs.head(1) if len(artist_songs) > 1 else artist_songs

ranked_recs_tfidf = get_ranked_recommendations(liked_songs, n_recs=10, model_type="tf-idf", semantic_weight=0.7)
ranked_recs_semantic = get_ranked_recommendations(liked_songs, n_recs=10, model_type="semantic", semantic_weight=0.7)

print(f"Artist: {artist}")
display(liked_songs["Title"])

print("Semantic + Feature Recommendations:")
display(ranked_recs_semantic)

print("TF-IDF + Feature Recomendation")
display(ranked_recs_tfidf)

Artist: drake


14    in my feelings
Name: Title, dtype: object

Semantic + Feature Recommendations:


,Rank,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,191,in my feelings,drake,NaN,0.835,0.626,-5833.0,0.125,0.059,0.0,0.350,0.574377,1.000000,0.872313
1,184,sin (feat. jaden smith),young thug,jaden smith,0.889,0.385,-11767.0,0.165,0.005,0.0,0.235,0.950040,0.638506,0.731966
2,200,in my head,lil tjay,NaN,0.684,0.545,-6885.0,0.121,0.159,0.0,0.407,0.920305,0.649714,0.730892
3,177,ztfo,big sean,NaN,0.720,0.542,-7583.0,0.048,0.051,0.0,0.374,0.830682,0.679302,0.724716
4,190,incredible,future,NaN,0.790,0.438,-6462.0,0.133,0.063,0.0,0.247,0.825821,0.668945,0.716008
5,102,perplexing pegasus,rae sremmurd,NaN,0.898,0.447,-8849.0,0.077,0.001,0.0,0.308,0.865657,0.647107,0.712672
6,196,perplexing pegasus,rae sremmurd,NaN,0.899,0.448,-8789.0,0.074,0.001,0.0,0.302,0.861707,0.647107,0.711487
7,177,afterglow,taylor swift,NaN,0.756,0.449,-8746.0,0.034,0.130,0.0,0.399,0.818617,0.664955,0.711053
8,157,"12.38 (feat. 21 savage, ink & kadhja bonet)",childish gambino,"21 savage, ink, kadhja bonet",0.691,0.524,-9718.0,0.160,0.197,0.0,0.410,0.863496,0.642415,0.708739
9,198,falling,trevor daniel,NaN,0.784,0.430,-8756.0,0.036,0.123,0.0,0.236,0.850480,0.646497,0.707692


TF-IDF + Feature Recomendation


,Rank,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Feature Similarity,Lyric Similarity,Combined Score
0,191,in my feelings,drake,NaN,0.835,0.626,-5833.0,0.125,0.059,0.0,0.350,0.574377,1.000000,0.872313
1,200,you got it,vedo,NaN,0.762,0.433,-8937.0,0.187,0.143,0.0,0.394,0.894402,0.152291,0.374924
2,172,pelle coat,lil durk,NaN,0.747,0.565,-8859.0,0.096,0.004,0.0,0.395,0.929012,0.127115,0.367684
3,157,"12.38 (feat. 21 savage, ink & kadhja bonet)",childish gambino,"21 savage, ink, kadhja bonet",0.691,0.524,-9718.0,0.160,0.197,0.0,0.410,0.863496,0.142754,0.358976
4,200,in my head,lil tjay,NaN,0.684,0.545,-6885.0,0.121,0.159,0.0,0.407,0.920305,0.115768,0.357129
5,182,bëttr 0ff,yeat,NaN,0.731,0.532,-9815.0,0.074,0.002,0.0,0.370,0.933177,0.107010,0.354860
6,184,sin (feat. jaden smith),young thug,jaden smith,0.889,0.385,-11767.0,0.165,0.005,0.0,0.235,0.950040,0.099567,0.354709
7,200,monëy so big,yeat,NaN,0.693,0.575,-7633.0,0.091,0.121,0.0,0.353,0.901935,0.119779,0.354426
8,192,mean it,"lany, lauv",NaN,0.746,0.450,-8543.0,0.087,0.041,0.0,0.336,0.948875,0.097482,0.352900
9,195,mean it,"lany, lauv",NaN,0.746,0.450,-8543.0,0.087,0.041,0.0,0.336,0.948875,0.097482,0.352900


In [10]:
# Quantitative Methods to Compare TF-IDF vs Semantic Embeddings
def evaluate_model_quality(model_type='semantic', n_samples=500, hybrid_tfidf_weight=0.5):
    # Get embeddings based on model type
    if model_type == 'semantic':
        emb_dense = ensure_semantic_embeddings()
    elif model_type == 'tfidf':
        emb = embeddings  # TF-IDF embeddings (sparse)
        emb_dense = emb.toarray() if hasattr(emb, 'toarray') else emb
    elif model_type == 'hybrid':
        # Sample first, then create hybrid embeddings for sample only
        sample_indices = random.sample(range(len(data)), min(n_samples, len(data)))
        sample_data = data.iloc[sample_indices].copy()
        emb_dense = get_hybrid_embeddings_sample(
            sample_indices,
            tfidf_weight=hybrid_tfidf_weight, 
            semantic_weight=1.0 - hybrid_tfidf_weight
        )
        # Set sample_embeddings directly since we already have the right data
        sample_embeddings = emb_dense
    elif model_type == 'features':
        # Use only audio features as embeddings
        feature_data = data[features].fillna(data[features].median())
        scaler = StandardScaler()
        emb_dense = scaler.fit_transform(feature_data)
    elif model_type == 'random':
        # Use random embeddings as baseline
        np.random.seed(42)  # For reproducibility
        emb_dense = np.random.randn(len(data), 100)  # 100-dim random vectors
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
    
    # Sample random songs for evaluation (hybrid handled separately above)
    if model_type != 'hybrid':
        sample_indices = random.sample(range(len(data)), min(n_samples, len(data)))
        sample_data = data.iloc[sample_indices].copy()
        sample_embeddings = emb_dense[sample_indices]
    
    results = {}
    
    # 1. Silhouette Score (How well separated are the clusters?)
    n_clusters = 20
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(sample_embeddings)
    
    silhouette_avg = silhouette_score(sample_embeddings, cluster_labels)
    results['silhouette_score'] = silhouette_avg
    
    # 3. Recommendation Consistency (do recommended songs have similar audio features?)
    consistency_scores = []
    
    for i in range(min(50, len(sample_indices))):  # Test 50 songs
        song_idx = sample_indices[i]
        
        # Get recommendations using this model
        if model_type == 'tfidf':
            recs_df = get_ranked_recommendations(
                data.iloc[[song_idx]], n_recs=10, model_type='tf-idf', semantic_weight=0.0
            )
        elif model_type == 'semantic':
            recs_df = get_ranked_recommendations(
                data.iloc[[song_idx]], n_recs=10, model_type='semantic', semantic_weight=1.0
            )
        elif model_type == 'hybrid':
            recs_df = get_ranked_recommendations(
                data.iloc[[song_idx]], n_recs=10, model_type='hybrid', 
                hybrid_tfidf_weight=hybrid_tfidf_weight
            )
        elif model_type == 'features':
            recs_df = get_ranked_recommendations(
                data.iloc[[song_idx]], n_recs=10, model_type='features', semantic_weight=0.0
            )
        elif model_type == 'random':
            recs_df = get_ranked_recommendations(
                data.iloc[[song_idx]], n_recs=10, model_type='random', semantic_weight=0.0
            )
        
        if len(recs_df) > 5:
            # Check if top recommendations have similar audio features
            original_features = data.iloc[song_idx][features].values
            rec_features = recs_df[features].head(5).values
            similarities = []
            for rec_feat in rec_features:
                sim = cosine_similarity([original_features], [rec_feat])[0][0]
                similarities.append(sim)
            
            feature_consistency = np.mean(similarities)
            consistency_scores.append(feature_consistency)
    
    avg_consistency = np.mean(consistency_scores) if consistency_scores else 0
    results['recommendation_consistency'] = avg_consistency
    
    # 4. Semantic Coherence Test (do similar artists cluster together?)
    artist_coherence_scores = []
    unique_artists = sample_data['Primary Artists'].value_counts()
    
    for artist, count in unique_artists.head(10).items():  # Top 10 artists
        if count >= 2:  # Artist has at least 2 songs
            artist_songs = sample_data[sample_data['Primary Artists'] == artist]
            artist_indices = [sample_indices.index(idx) for idx in artist_songs.index if idx in sample_indices]
            
            if len(artist_indices) >= 2:
                artist_embs = sample_embeddings[artist_indices]
                # Calculate average pairwise distance within artist
                distances = pairwise_distances(artist_embs, metric='cosine')
                avg_intra_distance = np.mean(distances[np.triu_indices(len(distances), k=1)])
                artist_coherence_scores.append(1 - avg_intra_distance)  # Convert to similarity
    
    avg_artist_coherence = np.mean(artist_coherence_scores) if artist_coherence_scores else 0
    results['artist_coherence'] = avg_artist_coherence
    
    # 5. Diversity (how varied are the recommendations?)
    
    # Calculate average pairwise distances
    sample_distances = pairwise_distances(sample_embeddings, metric='cosine')
    avg_distance = np.mean(sample_distances[np.triu_indices(len(sample_distances), k=1)])
    results['average_diversity'] = avg_distance
    
    # Calculate embedding dimensionality effectiveness
    explained_variance = np.var(sample_embeddings, axis=0).sum()
    results['embedding_variance'] = explained_variance
    
    return results

evaluation_complete = False

In [28]:
evaluation_complete = False

In [29]:
# Run evaluation for all models - prevent duplicate execution
if 'evaluation_complete' not in globals() or not evaluation_complete:
    n_samples = data.shape[0]
    print(f"Evaluating {n_samples} random songs from the dataset...")

    print(" - Getting TF-IDF Results...")
    tfidf_results = evaluate_model_quality('tfidf', n_samples=n_samples)
    print(" - Getting Semantic Results...")
    semantic_results = evaluate_model_quality('semantic', n_samples=n_samples)
    print(" - Getting Hybrid Results")
    # Evaluate hybrid across weights and pick best (primary: silhouette_score, tiebreaker: recommendation_consistency)
    hybrid_results_by_weight = {}
    best_weight = None
    best_score = (float('-inf'), float('-inf'))  # (silhouette_score, recommendation_consistency)

    for w in sorted({0.25, 0.5, 0.75}):
        print(f"    - Testing hybrid weight={w:.2f}...")
        res = evaluate_model_quality('hybrid', n_samples=n_samples, hybrid_tfidf_weight=w)
        hybrid_results_by_weight[w] = res

        sil = res.get('silhouette_score', float('-inf'))
        rc = float(res.get('recommendation_consistency', 0))
        score = (sil, rc)
        if score > best_score:
            best_score = score
            best_weight = w

    # Use best hybrid result
    hybrid_results = hybrid_results_by_weight[best_weight]
    print(f" - Best Hybrid weight: {best_weight:.1f} -> silhouette={best_score[0]:.4f}, recommendation_consistency={best_score[1]:.4f}")

        
    print(" - Getting Features-Only Results...")
    features_results = evaluate_model_quality('features', n_samples=n_samples)
    print(" - Getting Random Baseline Results...")
    random_results = evaluate_model_quality('random', n_samples=n_samples)
    
    evaluation_complete = True
else:
    print("Evaluation already complete. Using cached results.")

Evaluating 8034 random songs from the dataset...
 - Getting TF-IDF Results...


 - Getting Semantic Results...
 - Getting Hybrid Results
    - Testing hybrid weight=0.25...
 - Getting Hybrid Results
    - Testing hybrid weight=0.25...
    - Testing hybrid weight=0.50...
    - Testing hybrid weight=0.50...
    - Testing hybrid weight=0.75...
    - Testing hybrid weight=0.75...
 - Best Hybrid weight: 0.2 -> silhouette=0.0320, recommendation_consistency=0.9988
 - Getting Features-Only Results...
 - Best Hybrid weight: 0.2 -> silhouette=0.0320, recommendation_consistency=0.9988
 - Getting Features-Only Results...
 - Getting Random Baseline Results...
 - Getting Random Baseline Results...


In [30]:
# Mapping for display names and formatting rules (reuse existing `metrics` if present)
metrics = globals().get('metrics', [
    ('silhouette_score', 'Silhouette Score', '{:.4f}'),
    ('recommendation_consistency', 'Average Recommendation Consistency', '{:.4f}'),
    ('artist_coherence', 'Artist Coherence Score', '{:.4f}'),
    ('average_diversity', 'Average Pairwise Distance', '{:.4f}'),
    ('embedding_variance', 'Total Embedding Variance', '{:.2f}')
])

# Use existing ANSI variables if present, otherwise fall back
GREEN = '\033[92m'
WHITE = '\033[97m'
GREY = '\033[90m'
BOLD = '\033[1m'
RESET = '\033[0m'

# Display results table
if 'tfidf_results' in globals():
    header = f"{'\nMetric':40s} {'TF-IDF':12s} {'Semantic':12s} {'Hybrid':12s} {'Features':12s} {'Random':12s} {'Best'}"
    print(header)
    print("-" * len(header))

for key, display_name, _ in metrics:
    tfidf_val = tfidf_results.get(key, 0)
    semantic_val = semantic_results.get(key, 0)
    hybrid_val = hybrid_results.get(key, 0)
    features_val = features_results.get(key, 0)
    random_val = random_results.get(key, 0)
    
    # Format all values
    fmt = next(fmt for k, _, fmt in metrics if k == key)
    tfidf_str = fmt.format(tfidf_val)
    semantic_str = fmt.format(semantic_val)
    hybrid_str = fmt.format(hybrid_val)
    features_str = fmt.format(features_val)
    random_str = fmt.format(random_val)
    
    # Find the best value
    values = [tfidf_val, semantic_val, hybrid_val, features_val, random_val]
    names = ['TF-IDF', 'Semantic', 'Hybrid', 'Features', 'Random']
    if key == 'average_diversity':
        best_idx = values.index(min(values))
    else:
        best_idx = values.index(max(values))
    best_name = names[best_idx]
    
    # Color by rank: best=GREEN+BOLD, 2nd=DIM_GREEN, else=GREY
    raw_displays = [tfidf_str, semantic_str, hybrid_str, features_str, random_str]
    # Determine ranking (higher is better except for average_diversity where lower is better)
    reverse = (key != 'average_diversity')
    sorted_indices = sorted(range(len(values)), key=lambda i: values[i], reverse=reverse)
    
    displays = []
    for i, val in enumerate(raw_displays):
        # find rank of this column (0 = best)
        rank = sorted_indices.index(i)
        if rank == 0:
            color = GREEN
            prefix = BOLD
        elif rank == 1:
            color = WHITE
            prefix = ''
        else:
            color = GREY
            prefix = ''
        displays.append(f"{prefix}{color}{val:12s}{RESET}")
    
    print(f"{display_name:40s} {" ".join(displays)} {best_name}")

# Model Types:
#  - TF-IDF: Term Frequency-Inverse Document Frequency on lyrics
#  - Semantic: Sentence-transformer embeddings (all-MiniLM-L6-v2) on lyrics  
#  - Hybrid: Blend of TF-IDF and Semantic scores for Lyrics
#  - Features: Audio features only (Danceability, Energy, etc.)
#  - Random: Random embeddings (baseline control)
#
# Metrics:
#  - Silhouette Score: Measures how well-separated the clusters are in embedding space
#  - Recommendation Consistency: Measures how similar the song features are between recommendations
#  - Artist Coherence: Measures whether songs by the same artist cluster together
#  - Pairwise Distance: Measures how spread out/diverse songs are in the embedding space
#  - Embedding Variance: Total information captured across all embedding dimensions



Metric                                  TF-IDF       Semantic     Hybrid       Features     Random       Best
--------------------------------------------------------------------------------------------------------------
Silhouette Score                         0.0198       0.0268       0.0320       0.1403       0.0073       Features
Average Recommendation Consistency       0.9989       0.9782       0.9988       0.9942       0.9966       TF-IDF
Artist Coherence Score                   0.0699       0.4234       0.4159       0.2293       0.0003       Semantic
Average Pairwise Distance                0.9680       0.6725       0.6727       0.9879       1.0000       Semantic
Total Embedding Variance                 0.97         0.67         0.67         7.00         100.07       Random
